# DeepCare 02: Preprocessing and data split

Turns the raw CDC Diabetes Health Indicators data (cached by `01_eda.ipynb`) into leakage-free train/validation/test sets.

**Decisions made in this notebook**
1. Drop exact duplicate rows *before* splitting, so identical records cannot land in both train and test.
2. Stratified 70/15/15 split with a fixed seed.
3. Clip BMI at the 0.5th/99.5th percentiles of the **training set only**.
4. Standardize non-binary features; the scaler is fit on the **training set only**. Binary columns stay as 0/1.

**Outputs saved to Drive:** raw and scaled train/val/test CSVs, `scaler.joblib`, `prep_config.json`.

## 1. Load and deduplicate

In [1]:
import json, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
PROJECT_DIR = '/content/drive/MyDrive/deepcare'
TARGET = 'Diabetes_binary'

df_raw = pd.read_csv(f'{PROJECT_DIR}/data/raw.csv')
df = df_raw.drop_duplicates().reset_index(drop=True)

print('raw:', df_raw.shape, '| positive rate', round(df_raw[TARGET].mean(), 4))
print('dedup:', df.shape, '| positive rate', round(df[TARGET].mean(), 4))   # expect (229474, 22) and ~0.1529

Mounted at /content/drive
raw: (253680, 22) | positive rate 0.1393
dedup: (229474, 22) | positive rate 0.1529


## 2. Stratified train / validation / test split

70% train, 15% validation, 15% test. Stratifying on the target keeps the positive rate nearly identical in each part.

In [2]:
X, y = df.drop(columns=[TARGET]), df[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

for name, part in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{name:5s} rows={len(part):7,d}  positive rate={part.mean():.4f}')

train rows=160,631  positive rate=0.1529
val   rows= 34,421  positive rate=0.1529
test  rows= 34,422  positive rate=0.1530


## 3. Fit preprocessing on the training set only

- `BMI` is clipped at training-set percentiles (it has a thin tail up to 98).
- Ordinal/count columns (`BMI`, `MentHlth`, `PhysHlth`, `GenHlth`, `Age`, `Education`, `Income`) are standardized.
- Binary columns are left as 0/1.
- The output column order is fixed (`BIN + NUM`) and is reused by the demo app.

In [3]:
NUM = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
BIN = [c for c in X.columns if c not in NUM]

lo, hi = X_train['BMI'].quantile([0.005, 0.995])

def clip_bmi(d):
    d = d.copy()
    d['BMI'] = d['BMI'].clip(lo, hi)
    return d

scaler = StandardScaler().fit(clip_bmi(X_train)[NUM])

def transform(d):
    out = clip_bmi(d)
    out[NUM] = scaler.transform(out[NUM])
    return out[BIN + NUM]

Xtr, Xva, Xte = transform(X_train), transform(X_val), transform(X_test)

print(f'BMI clip bounds (train): {lo:.1f} to {hi:.1f}')
print(Xtr[NUM].agg(['mean', 'std']).round(3))   # train: mean ~0, std ~1

BMI clip bounds (train): 17.0 to 56.0
      BMI  MentHlth  PhysHlth  GenHlth  Age  Education  Income
mean -0.0       0.0       0.0     -0.0  0.0        0.0    -0.0
std   1.0       1.0       1.0      1.0  1.0        1.0     1.0


## 4. Leakage checks and saving

- **Exact-row overlap** (features + label) between splits should be 0, because duplicates were removed before splitting.
- **Feature-profile overlap** (features only) can be non-zero: it comes from the profiles that appear with conflicting labels. This is informational, not an error.

In [4]:
def row_set(Xd, yd):
    return set(map(tuple, pd.concat([Xd, yd], axis=1).to_numpy()))

a, b, c = row_set(X_train, y_train), row_set(X_val, y_val), row_set(X_test, y_test)
print('exact-row overlap  train&val:', len(a & b), '| train&test:', len(a & c), '| val&test:', len(b & c))  # expect 0

train_profiles = set(map(tuple, X_train.to_numpy()))
test_in_train = sum(tuple(r) in train_profiles for r in X_test.to_numpy())
print(f'test rows whose feature profile also occurs in train: {test_in_train} of {len(X_test)}')

exact-row overlap  train&val: 0 | train&test: 0 | val&test: 0
test rows whose feature profile also occurs in train: 326 of 34422


In [5]:
d = f'{PROJECT_DIR}/data'
for name, Xs, Xr, ys in [('train', Xtr, X_train, y_train),
                         ('val',   Xva, X_val,   y_val),
                         ('test',  Xte, X_test,  y_test)]:
    pd.concat([Xs, ys], axis=1).to_csv(f'{d}/{name}_scaled.csv', index=False)
    pd.concat([Xr[BIN + NUM], ys], axis=1).to_csv(f'{d}/{name}_raw.csv', index=False)

joblib.dump(scaler, f'{PROJECT_DIR}/scaler.joblib')
with open(f'{PROJECT_DIR}/prep_config.json', 'w') as f:
    json.dump({'NUM': NUM, 'BIN': BIN, 'feature_order': BIN + NUM,
               'bmi_clip': [float(lo), float(hi)], 'seed': SEED}, f, indent=2)

print('saved to', d)

saved to /content/drive/MyDrive/deepcare/data
